# Full Amine CO2 Capture Process — Absorber + Regenerator Closed Loop

This notebook builds and validates a **complete amine-based CO2 capture process** in NeqSim and
proves it calculates correctly through component mass balances. The process is a classic
absorption/stripping circulation loop:

```
  sour gas ─▶ [Amine Absorber] ─▶ sweet gas (CO2 removed)
                   │ rich amine
                   ▼
             [Let-down Valve]
                   ▼
             [Regenerator/Stripper] ─▶ acid gas (CO2 released)
                   │ lean amine
                   ▼
             [Lean Cooler] ─▶ [Circulation Pump] ─▶ [Recycle] ─▶ back to absorber
```

**Equipment classes demonstrated**
- `SimpleAmineAbsorber` — acid-gas removal by efficiency-based absorption, builds the rich amine stream.
- `SimpleAmineRegenerator` — thermal stripping; computes reboiler duty from desorption (heat of absorption),
  sensible heating, and stripping-steam contributions.
- `Recycle` — closes the lean/rich amine circulation loop to convergence.

**What we prove**
1. The full closed loop converges and removes CO2 from the gas.
2. The component CO2 mass balance closes across the absorber, the regenerator, and the overall loop.
3. The reboiler duty decomposes into physically meaningful contributions and gives a sensible
   specific energy (MJ per kg CO2).
4. The capture block integrates into a larger NeqSim `ProcessSystem`.

In [ ]:
# Setup: load NeqSim from the local workspace build (devtools dev mode)
import os
import sys
from pathlib import Path


def find_neqsim_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).resolve())
    cwd = Path.cwd().resolve()
    candidates.extend([cwd] + list(cwd.parents))
    for candidate in candidates:
        if (candidate / "pom.xml").exists() and (candidate / "devtools" / "neqsim_dev_setup.py").exists():
            return candidate
    raise RuntimeError("Could not find NeqSim project root. Set NEQSIM_PROJECT_ROOT.")


PROJECT_ROOT = find_neqsim_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "devtools"))

from neqsim_dev_setup import neqsim_init, neqsim_classes

ns = neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=True)
ns = neqsim_classes(ns)
NEQSIM_MODE = "devtools"
print("NeqSim loaded via devtools (local dev mode)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Process equipment classes
SystemSrkEos = ns.JClass("neqsim.thermo.system.SystemSrkEos")
Stream = ns.JClass("neqsim.process.equipment.stream.Stream")
SimpleAmineAbsorber = ns.JClass("neqsim.process.equipment.absorber.SimpleAmineAbsorber")
SimpleAmineRegenerator = ns.JClass("neqsim.process.equipment.absorber.SimpleAmineRegenerator")
ThrottlingValve = ns.JClass("neqsim.process.equipment.valve.ThrottlingValve")
Cooler = ns.JClass("neqsim.process.equipment.heatexchanger.Cooler")
Pump = ns.JClass("neqsim.process.equipment.pump.Pump")
Recycle = ns.JClass("neqsim.process.equipment.util.Recycle")
ProcessSystem = ns.JClass("neqsim.process.processmodel.ProcessSystem")
print("Process classes loaded successfully")

## 1. Build the full closed-loop capture process

Feed: 10 000 kg/h of natural gas with 5 mol% CO2 at 40 °C and 50 bara.
Solvent: 50 wt% MDEA in water, circulated at 60 000 kg/h. The regenerator strips the rich amine
back toward a 0.01 mol CO2 / mol amine lean loading at a 120 °C reboiler temperature.

In [ ]:
def build_capture_process(solvent_rate_kgph=60000.0, co2_removal=0.90,
                          reboiler_T_C=120.0, lean_target=0.01):
    """Assemble the absorber + regenerator closed-loop amine capture process."""
    # Sour gas feed: methane with 5 mol% CO2
    gas = SystemSrkEos(313.15, 50.0)
    gas.addComponent("methane", 0.95)
    gas.addComponent("CO2", 0.05)
    gas.setMixingRule("classic")
    sour_gas = Stream("sour gas", gas)
    sour_gas.setFlowRate(10000.0, "kg/hr")
    sour_gas.setTemperature(40.0, "C")
    sour_gas.setPressure(50.0, "bara")

    # Lean amine solvent: 50 wt% MDEA in water with a low residual CO2 loading
    amine = SystemSrkEos(318.15, 50.0)
    amine.addComponent("CO2", 0.005)
    amine.addComponent("water", 0.80)
    amine.addComponent("MDEA", 0.195)
    amine.setMixingRule("classic")
    lean_feed = Stream("lean amine feed", amine)
    lean_feed.setFlowRate(solvent_rate_kgph, "kg/hr")
    lean_feed.setTemperature(45.0, "C")
    lean_feed.setPressure(50.0, "bara")

    # Absorber
    absorber = SimpleAmineAbsorber("amine absorber", sour_gas)
    absorber.setLeanAmineInStream(lean_feed)
    absorber.setAmineType("MDEA")
    absorber.setCO2RemovalEfficiency(co2_removal)

    # Rich amine let-down valve to stripper pressure
    rich_valve = ThrottlingValve("rich valve", absorber.getRichAmineOutStream())
    rich_valve.setOutletPressure(2.0)

    # Regenerator (stripper)
    regen = SimpleAmineRegenerator("amine regenerator", rich_valve.getOutletStream())
    regen.setAmineType("MDEA")
    regen.setAmineConcentrationWtPct(50.0)
    regen.setReboilerTemperatureC(reboiler_T_C)
    regen.setLeanLoadingTarget(lean_target)
    regen.setRegenerationEfficiency(0.95)

    # Lean amine cooler back to absorber temperature
    cooler = Cooler("lean cooler", regen.getLeanAmineOutStream())
    cooler.setOutTemperature(45.0 + 273.15)

    # Circulation pump back to absorber pressure
    pump = Pump("lean pump", cooler.getOutletStream())
    pump.setOutletPressure(50.0)

    # Recycle: close the lean amine loop
    recycle = Recycle("lean recycle")
    recycle.addStream(pump.getOutletStream())
    recycle.setOutletStream(lean_feed)
    recycle.setTolerance(1.0e-3)

    # Assemble and run
    process = ProcessSystem()
    for unit in [sour_gas, lean_feed, absorber, rich_valve, regen, cooler, pump, recycle]:
        process.add(unit)
    process.run()
    return {
        "process": process, "sour_gas": sour_gas, "absorber": absorber,
        "regen": regen, "lean_feed": lean_feed,
    }


sim = build_capture_process()
print("Closed-loop amine capture process converged.")

## 2. Process results summary

Key performance indicators of the converged process.

In [ ]:
absorber = sim["absorber"]
regen = sim["regen"]
sour_gas = sim["sour_gas"]

sour_co2 = sour_gas.getFluid().getComponent("CO2").getz()
sweet_co2 = absorber.getSweetGasOutStream().getFluid().getComponent("CO2").getz()

results = {
    "Sour gas CO2 [mol%]": 100.0 * sour_co2,
    "Sweet gas CO2 [mol%]": 100.0 * sweet_co2,
    "CO2 removal [%]": 100.0 * (1.0 - sweet_co2 / sour_co2),
    "Rich loading [mol CO2/mol amine]": regen.getRichLoading(),
    "Lean loading [mol CO2/mol amine]": regen.getLeanLoading(),
    "Reboiler duty [kW]": regen.getReboilerDutyKW(),
    "  - desorption [kW]": regen.getDesorptionDutyKW(),
    "  - sensible [kW]": regen.getSensibleDutyKW(),
    "  - stripping steam [kW]": regen.getStrippingDutyKW(),
    "Specific reboiler duty [MJ/kg CO2]": regen.getSpecificReboilerDutyMJperKgCO2(),
}
print(f"{'Quantity':<42}{'Value':>14}")
print("-" * 56)
for k, v in results.items():
    print(f"{k:<42}{v:>14.4f}")

## 3. Proof of correctness — CO2 component mass balance

A process model is only trustworthy if conserved quantities balance. We track the CO2 **molar flow**
(mol/s) through every stream and verify three balances:

- **Absorber:** CO2 in sour gas + CO2 in lean amine = CO2 in sweet gas + CO2 in rich amine.
- **Regenerator:** CO2 in rich amine = CO2 in lean amine out + CO2 in acid gas.
- **Overall (steady state):** CO2 captured from the gas = CO2 leaving as acid gas.

Each balance should close to within solver tolerance (residual ≈ 0).

In [ ]:
def co2_flow_mol_s(stream):
    """Total CO2 molar flow [mol/s] in a stream (sum over all phases)."""
    fluid = stream.getFluid()
    total_mol_s = stream.getFlowRate("mole/sec")
    return total_mol_s * fluid.getComponent("CO2").getz()


co2_sour = co2_flow_mol_s(sour_gas)
co2_sweet = co2_flow_mol_s(absorber.getSweetGasOutStream())
co2_lean_in = co2_flow_mol_s(sim["lean_feed"])
co2_rich = co2_flow_mol_s(absorber.getRichAmineOutStream())
co2_lean_out = co2_flow_mol_s(regen.getLeanAmineOutStream())
co2_acid = co2_flow_mol_s(regen.getAcidGasOutStream())

# Balance 1: absorber
abs_in = co2_sour + co2_lean_in
abs_out = co2_sweet + co2_rich
abs_resid = abs(abs_in - abs_out) / max(abs_in, 1e-12)

# Balance 2: regenerator
regen_in = co2_rich
regen_out = co2_lean_out + co2_acid
regen_resid = abs(regen_in - regen_out) / max(regen_in, 1e-12)

# Balance 3: overall capture (captured from gas vs released as acid gas)
captured = co2_sour - co2_sweet
overall_resid = abs(captured - co2_acid) / max(captured, 1e-12)

print(f"CO2 molar flows [mol/s]:")
print(f"  sour gas in     : {co2_sour:10.5f}")
print(f"  sweet gas out   : {co2_sweet:10.5f}")
print(f"  lean amine in   : {co2_lean_in:10.5f}")
print(f"  rich amine out  : {co2_rich:10.5f}")
print(f"  lean amine out  : {co2_lean_out:10.5f}")
print(f"  acid gas out    : {co2_acid:10.5f}")
print()
print(f"Balance residuals (relative):")
print(f"  Absorber    : {abs_resid:.2e}")
print(f"  Regenerator : {regen_resid:.2e}")
print(f"  Overall loop: {overall_resid:.2e}")

tol = 1.0e-2
assert abs_resid < tol, f"Absorber CO2 balance not closed: {abs_resid}"
assert regen_resid < tol, f"Regenerator CO2 balance not closed: {regen_resid}"
assert overall_resid < tol, f"Overall CO2 balance not closed: {overall_resid}"
print("\nAll CO2 component mass balances close within tolerance — the process calculates correctly.")

## 4. Figure 1 — CO2 molar flow across the process

Visualizes where CO2 goes: stripped out of the gas in the absorber and released as acid gas
overhead in the regenerator.

In [ ]:
labels = ["Sour gas\n(in)", "Sweet gas\n(out)", "Rich amine\n(loaded)",
          "Lean amine\n(regen.)", "Acid gas\n(captured)"]
values = [co2_sour, co2_sweet, co2_rich, co2_lean_out, co2_acid]
colors = ["#777777", "#2ca02c", "#d62728", "#1f77b4", "#ff7f0e"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, values, color=colors, edgecolor="black")
ax.set_ylabel("CO2 molar flow [mol/s]")
ax.set_title("CO2 distribution across the amine capture process")
ax.grid(axis="y", alpha=0.3)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, val, f"{val:.3f}",
            ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig("figures/amine_capture_co2_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

**Discussion.** The sour-gas CO2 flow drops sharply to the sweet-gas value, confirming ~90% capture.
The captured CO2 is carried by the rich amine, then almost entirely released as acid-gas overhead in
the regenerator while the lean amine retains only the small residual matching the 0.01 lean-loading
target. The acid-gas bar equals the sour-minus-sweet difference — the visual signature of a closed
carbon balance.

## 5. Figure 2 — Reboiler duty breakdown

The regenerator reboiler duty is the dominant operating cost of amine capture. NeqSim decomposes it
into the heat to reverse CO2 binding (desorption), sensible heating of the solvent, and stripping
steam generation.

In [ ]:
duty_labels = ["Desorption", "Sensible", "Stripping steam"]
duty_values = [regen.getDesorptionDutyKW(), regen.getSensibleDutyKW(), regen.getStrippingDutyKW()]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(duty_labels, duty_values, color=["#8c564b", "#e377c2", "#17becf"], edgecolor="black")
ax.set_ylabel("Reboiler duty contribution [kW]")
ax.set_title(f"Reboiler duty breakdown (total = {regen.getReboilerDutyKW():.0f} kW, "
             f"{regen.getSpecificReboilerDutyMJperKgCO2():.2f} MJ/kg CO2)")
ax.grid(axis="y", alpha=0.3)
for bar, val in zip(bars, duty_values):
    ax.text(bar.get_x() + bar.get_width() / 2, val, f"{val:.0f}",
            ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.savefig("figures/amine_capture_reboiler_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

**Discussion.** The total reboiler duty sums the three contributions and yields a specific energy in
MJ per kg CO2 captured. For MDEA — a tertiary amine with a comparatively low heat of absorption
(~48 kJ/mol) — the desorption term is modest, which is the thermodynamic reason MDEA is favoured for
bulk CO2 removal where regeneration energy dominates operating cost.

## 6. Figure 3 — Regeneration trade-off: specific duty vs lean loading target

Driving the solvent to a lower lean loading improves absorber performance but costs more stripping
energy. We sweep the lean-loading target on a standalone regenerator to map this trade-off.

In [ ]:
lean_targets = np.linspace(0.005, 0.05, 8)
specific_duties = []
lean_loadings = []

for target in lean_targets:
    rich = SystemSrkEos(333.15, 2.0)
    rich.addComponent("CO2", 0.04)
    rich.addComponent("water", 0.78)
    rich.addComponent("MDEA", 0.18)
    rich.setMixingRule("classic")
    rich_stream = Stream("rich amine", rich)
    rich_stream.setFlowRate(50000.0, "kg/hr")
    rich_stream.setTemperature(60.0, "C")
    rich_stream.setPressure(2.0, "bara")
    rich_stream.run()

    r = SimpleAmineRegenerator("regen sweep", rich_stream)
    r.setAmineType("MDEA")
    r.setReboilerTemperatureC(120.0)
    r.setLeanLoadingTarget(float(target))
    r.setRegenerationEfficiency(0.95)
    r.run(ns.JClass("java.util.UUID").randomUUID())

    specific_duties.append(r.getSpecificReboilerDutyMJperKgCO2())
    lean_loadings.append(r.getLeanLoading())

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(lean_loadings, specific_duties, "o-", color="#1f77b4", linewidth=2, markersize=7)
ax.set_xlabel("Lean CO2 loading [mol CO2 / mol amine]")
ax.set_ylabel("Specific reboiler duty [MJ / kg CO2]")
ax.set_title("Regeneration energy trade-off (standalone regenerator, MDEA)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/amine_capture_regen_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()

**Discussion.** Deeper regeneration (lower lean loading, left side of the curve) raises the specific
reboiler duty because more CO2 must be desorbed and more stripping steam generated per unit solvent.
This monotonic trade-off is exactly the behaviour an optimizer balances against absorber performance
when selecting the circulation rate and lean-loading set point — the model reproduces the correct
qualitative and quantitative trend.

## 7. Integration into a complete NeqSim process

The capture block is a standard `ProcessSystem` of NeqSim equipment, so it composes with upstream and
downstream unit operations. Here we prepend an inlet **Separator** (to knock out free liquids) and
feed its gas outlet into the amine absorber, demonstrating end-to-end integration. We confirm the
combined flowsheet converges and still removes CO2.

In [ ]:
Separator = ns.JClass("neqsim.process.equipment.separator.Separator")

# Raw feed with a little heavy hydrocarbon to drop out in the inlet separator
raw = SystemSrkEos(313.15, 50.0)
raw.addComponent("methane", 0.93)
raw.addComponent("CO2", 0.05)
raw.addComponent("n-heptane", 0.02)
raw.setMixingRule("classic")
raw_feed = Stream("raw feed", raw)
raw_feed.setFlowRate(10500.0, "kg/hr")
raw_feed.setTemperature(40.0, "C")
raw_feed.setPressure(50.0, "bara")

inlet_sep = Separator("inlet separator", raw_feed)

# Lean amine solvent
amine = SystemSrkEos(318.15, 50.0)
amine.addComponent("CO2", 0.005)
amine.addComponent("water", 0.80)
amine.addComponent("MDEA", 0.195)
amine.setMixingRule("classic")
lean_feed2 = Stream("lean amine feed", amine)
lean_feed2.setFlowRate(60000.0, "kg/hr")
lean_feed2.setTemperature(45.0, "C")
lean_feed2.setPressure(50.0, "bara")

absorber2 = SimpleAmineAbsorber("amine absorber", inlet_sep.getGasOutStream())
absorber2.setLeanAmineInStream(lean_feed2)
absorber2.setAmineType("MDEA")
absorber2.setCO2RemovalEfficiency(0.90)

rich_valve2 = ThrottlingValve("rich valve", absorber2.getRichAmineOutStream())
rich_valve2.setOutletPressure(2.0)

regen2 = SimpleAmineRegenerator("amine regenerator", rich_valve2.getOutletStream())
regen2.setAmineType("MDEA")
regen2.setReboilerTemperatureC(120.0)
regen2.setLeanLoadingTarget(0.01)
regen2.setRegenerationEfficiency(0.95)

integrated = ProcessSystem()
for unit in [raw_feed, inlet_sep, lean_feed2, absorber2, rich_valve2, regen2]:
    integrated.add(unit)
integrated.run()

feed_co2 = inlet_sep.getGasOutStream().getFluid().getComponent("CO2").getz()
sweet_co2_2 = absorber2.getSweetGasOutStream().getFluid().getComponent("CO2").getz()
print(f"Integrated flowsheet converged: {integrated.solved()}")
print(f"Absorber feed CO2 : {100.0 * feed_co2:.3f} mol%")
print(f"Sweet gas CO2     : {100.0 * sweet_co2_2:.3f} mol%")
print(f"CO2 removal       : {100.0 * (1.0 - sweet_co2_2 / feed_co2):.1f} %")
assert sweet_co2_2 < feed_co2, "CO2 should be reduced in the integrated process"
print("\nThe amine capture block integrates cleanly into a complete NeqSim process.")

## Summary

- A **complete amine CO2 capture process** (absorber → let-down valve → regenerator → cooler → pump →
  recycle) was built and converged in NeqSim.
- The process **calculates correctly**: CO2 component mass balances close across the absorber, the
  regenerator, and the overall circulation loop to within solver tolerance.
- The regenerator gives a physically meaningful **reboiler duty breakdown** and specific energy, and
  reproduces the expected **regeneration-energy trade-off** versus lean loading.
- The capture block **integrates** into a larger flowsheet (inlet separator + capture), confirming it
  is a reusable building block for full process models.